# List-aware ensemble release candidate

This is a copy of the existing **off-market `ensemble_release.ipynb`** workflow, adapted for **list-aware** pricing.

The same stages remain: training-only BoW feature bags and row sampling, 700-tree boosting candidates, Optuna tuning, later-month marginal ensemble admission, restartable checkpoints, frozen shadow audit, and full-history model refit.

Changes from off-market: **`log(sold_price/list_price)`** instead of direct log sale price, sale-price reconstruction using known list price, and candidate estimators from **CatBoost, XGBoost, or LightGBM** with curated feature recipes.

Use **Colab/day**, **overnight/local** or **quiet/local** profiles as in the original notebook. Switching CPU thread profiles does not change experimental model definitions. One worker at a time per checkpoint. Training and the one-way audit are independently opt-in and **disabled by default**.

Previously studied August–October 2025 months are a chronological shadow audit, not an untouched holdout. The 3.8996% earlier benchmark used a different protocol, so compare only matched-cohort results.


## 1 · One-time Colab setup

As with the off-market notebook, run the setup cells on Colab only. Use local VM storage for the live Optuna SQLite database, and a Google Drive ZIP for periodic checkpoint backup. The friend running this should use the **same dataset** and compatible Python dependencies.


In [1]:
# Optional Google Drive mount on Colab only.
MOUNT_GOOGLE_DRIVE = True
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import sys
import subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/SSU-Data-Dash-hdnl-comp")
    if not repo.exists():
        subprocess.run([
            "git", "clone",
            "https://github.com/HiddenL123/SSU-Data-Dash-hdnl-comp.git",
            str(repo)
        ], check=True)
    else:
        # Refresh an existing Colab checkout; stop on conflicts.
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
    os.chdir(repo)
# Locate the checkout from Jupyter/VS Code, including nested notebook folders.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("DATA_DASH_ROOT", Path.cwd())).expanduser().resolve()
ROOT = next((p for p in (_start, *_start.parents)
             if (p / "data.py").is_file() and (p / "models").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set DATA_DASH_ROOT to your Data Dash checkout before running this notebook.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [3]:
# On Colab only, install the existing repository requirements and Optuna.
# Local VS Code kernels use the repository's own environment.
if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         str(ROOT / "requirements.txt"), "optuna", "catboost", "xgboost", "lightgbm"],
        check=True,
    )


In [4]:
import data

homes = data.load("listings")

print("Training homes:", homes.shape)
print("Columns:", len(homes.columns))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:140: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Training homes: (150000, 22)
Columns: 22


In [5]:
# Optional persistent checkpoint directory in Colab Drive.
if "google.colab" in sys.modules and MOUNT_GOOGLE_DRIVE:
    from pathlib import Path
    Path("/content/drive/MyDrive/automl").mkdir(parents=True, exist_ok=True)


## 2 · Choose where this session runs

Keep the **same fold protocol** when moving between Colab and local PC. The resource profile determines CPU threads only.

The list-aware factory uses Apr–May 2025 for tuning, Jun–Jul for ensemble admission, and Aug–Oct for the independently opened audit. The notebook invokes `models.listaware_ensemble_automl_factory` directly rather than the off-market CLI.


In [6]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

# Check the source version before importing the factory and its dependents.
import models.listaware_factory_search as factory_search
_required = ("MODEL_FAMILIES", "CATBOOST_OBJECTIVES", "TEXT_MODES",
             "ROW_MODES", "ADDON_OPTIONS", "COMPARABLE_OPTIONS")
_missing = [name for name in _required if not hasattr(factory_search, name)]
if _missing:
    raise RuntimeError(
        f"Outdated list-aware module: {factory_search.__file__}; missing {_missing}. "
        "Commit and push the matching local models/listaware_factory_search.py "
        "and models/listaware_ensemble_automl_factory.py changes to the cloned "
        "repository branch, rerun the checkout setup cell, then restart the "
        "Python session and run the notebook from the top. "
        "Installing pip packages will not fix this source-version mismatch."
    )

from models.automl_profiles import get_profile
from models.listaware_ensemble_automl_factory import (
    FactorySettings, FactoryRun,
)
from models.listaware_factory_search import (
    MODEL_FAMILIES,
    CATBOOST_OBJECTIVES,
    TEXT_MODES,
    ROW_MODES,
    ADDON_OPTIONS,
    COMPARABLE_OPTIONS,
)
from models.automl_portable import export_checkpoint, import_checkpoint

RELEASE_PROFILE = "overnight_local"  # "day_cloud", "overnight_local", "day_local_quiet"
RELEASE_OUTPUT_DIR = "outputs/listaware_ensemble_release_full"
RELEASE_GENERATIONS = 2             # Increase for a longer experiment BEFORE starting
RELEASE_CANDIDATES_PER_GENERATION = 3
RELEASE_OPTUNA_TRIALS = 5
RELEASE_CHECKPOINT_ARCHIVE = None   # e.g. "/content/drive/MyDrive/automl/listaware_checkpoint.zip"

profile = get_profile(RELEASE_PROFILE)
settings = FactorySettings(
    output_dir=RELEASE_OUTPUT_DIR,
    generations=RELEASE_GENERATIONS,
    candidates_per_generation=RELEASE_CANDIDATES_PER_GENERATION,
    optuna_trials=RELEASE_OPTUNA_TRIALS,
    trees=700,
    train_cap=200_000,
    validation_cap=100_000,
)
print("Execution:", profile)
print("Frozen list-aware settings:", settings)
print("Available model families:", MODEL_FAMILIES)
print("CatBoost objectives:", CATBOOST_OBJECTIVES)
print("Text bagging modes:", TEXT_MODES)
print("Row sampling modes:", ROW_MODES)
print("Spatial feature combinations:", ADDON_OPTIONS)
print("Comparable adjustments:", COMPARABLE_OPTIONS)


Execution: WorkerProfile(name='overnight_local', xgb_threads=8, description='Local full-speed CPU training when the computer is otherwise idle', kind='local')
Frozen list-aware settings: FactorySettings(tuning_months=('2025-04', '2025-05'), selection_months=('2025-06', '2025-07'), audit_months=('2025-08', '2025-09', '2025-10'), train_start='1900-01-01', train_cap=200000, validation_cap=100000, trees=700, max_extra_terms=1000, min_df=10, candidates_per_generation=3, generations=2, optuna_trials=5, minimum_ensemble_gain_pp=0.02, max_mape_regression_pp=0.03, max_members=8, seed=42, output_dir='outputs/listaware_ensemble_release_full', save_accepted_snapshots=False, screen_max_mdape=15.0)
Available model families: ('catboost', 'xgboost', 'lightgbm')
CatBoost objectives: ('MAE', 'RMSE', 'Huber:delta=0.05')
Text bagging modes: ('selected', 'selected_plus', 'weighted', 'uniform', 'full')
Row sampling modes: ('all', 'bootstrap', 'subbag')
Spatial feature combinations: ((), ('spacetime_splines'

In [7]:
# Run or RESUME using the original off-market factory workflow.
# Completed folds, Optuna trials, member decisions and checkpoint archives are reused.
RUN_FULL_RELEASE_FACTORY = True
if RUN_FULL_RELEASE_FACTORY:
    factory = FactoryRun(
        settings,
        runtime_threads=profile.xgb_threads,
        checkpoint_archive=RELEASE_CHECKPOINT_ARCHIVE,
    )
    factory_result = factory.run()
    print("Finished list-aware selection:", factory_result[0].get("selection_metrics"))
    print("Selected weights:", factory_result[0].get("effective_member_weights"))



[1/2] Trying reference_catboost: {'name': 'reference_catboost', 'seed': 42, 'family': 'catboost', 'objective': 'MAE', 'text_mode': 'selected', 'n_text': 150, 'selected_weight': 4.0, 'row_mode': 'all', 'row_fraction': 1.0, 'addons': ('spacetime_splines', 'geo_context'), 'comparable': 'base'}


[I 2026-10-25 22:37:13,847] A new study created in RDB with name: listaware_factory_v1_reference_catboost
[I 2026-10-25 22:48:42,038] Trial 0 finished with value: 4.929893638487028 and parameters: {'depth': 6, 'learning_rate': 0.04, 'l2_leaf_reg': 5}. Best is trial 0 with value: 4.929893638487028.
[I 2026-10-25 22:52:36,731] Trial 1 finished with value: 4.952998940314702 and parameters: {'depth': 4, 'learning_rate': 0.04970750323455676, 'l2_leaf_reg': 2.444603344079418}. Best is trial 0 with value: 4.929893638487028.
[I 2026-10-25 23:03:52,536] Trial 2 finished with value: 4.887875605979456 and parameters: {'depth': 8, 'learning_rate': 0.1570894109175486, 'l2_leaf_reg': 1.6188622243748512}. Best is trial 2 with value: 4.887875605979456.
[I 2026-10-25 23:06:29,590] Trial 3 pruned. 
[I 2026-10-25 23:08:25,898] Trial 4 pruned. 


  ADMIT gain=None ensemble size=1

[1/2] Trying reference_xgboost: {'name': 'reference_xgboost', 'seed': 43, 'family': 'xgboost', 'objective': 'RMSE', 'text_mode': 'selected', 'n_text': 150, 'selected_weight': 4.0, 'row_mode': 'all', 'row_fraction': 1.0, 'addons': ('spacetime_splines', 'geo_context'), 'comparable': 'base'}


[I 2026-10-25 23:39:53,140] A new study created in RDB with name: listaware_factory_v1_reference_xgboost
[I 2026-10-25 23:54:07,440] Trial 0 finished with value: 4.929821658919922 and parameters: {'max_depth': 8, 'min_child_weight': 16.684619386293466, 'learning_rate': 0.09522271657253033, 'reg_lambda': 19.50331236061207, 'subsample': 0.9804685587557792, 'colsample_bytree': 0.8994749947027713}. Best is trial 0 with value: 4.929821658919922.
[I 2026-10-26 00:04:21,347] Trial 1 finished with value: 4.894674533696593 and parameters: {'max_depth': 8, 'min_child_weight': 18.30440433394384, 'learning_rate': 0.049424794844362976, 'reg_lambda': 12.90578426879155, 'subsample': 0.8183766335880667, 'colsample_bytree': 0.8569378235050102}. Best is trial 1 with value: 4.894674533696593.
[I 2026-10-26 00:10:02,258] Trial 2 finished with value: 4.891552213590392 and parameters: {'max_depth': 6, 'min_child_weight': 13.555472181490043, 'learning_rate': 0.10262249480500556, 'reg_lambda': 12.647887481771

  reject gain=0.017481772759023073 ensemble size=1

[1/2] Trying reference_lightgbm: {'name': 'reference_lightgbm', 'seed': 44, 'family': 'lightgbm', 'objective': 'RMSE', 'text_mode': 'selected', 'n_text': 150, 'selected_weight': 4.0, 'row_mode': 'all', 'row_fraction': 1.0, 'addons': ('spacetime_splines', 'geo_context'), 'comparable': 'base'}


...

  reject gain=0.0049734957775235955 ensemble size=2

[2/2] Trying g001_c001_xgboost: {'name': 'g001_c001_xgboost', 'seed': 1273194232, 'family': 'xgboost', 'objective': 'RMSE', 'text_mode': 'uniform', 'n_text': 250, 'selected_weight': 4.0, 'row_mode': 'all', 'row_fraction': 1.0, 'addons': ('geo_context',), 'comparable': 'comp_age_first'}


[I 2026-10-26 02:52:52,651] A new study created in RDB with name: listaware_factory_v1_g001_c001_xgboost
[I 2026-10-26 03:04:34,920] Trial 0 finished with value: 5.049392796785737 and parameters: {'max_depth': 8, 'min_child_weight': 16.684619386293466, 'learning_rate': 0.09522271657253033, 'reg_lambda': 19.50331236061207, 'subsample': 0.9804685587557792, 'colsample_bytree': 0.8994749947027713}. Best is trial 0 with value: 5.049392796785737.
[I 2026-10-26 03:07:47,584] Trial 1 finished with value: 4.945551391399786 and parameters: {'max_depth': 4, 'min_child_weight': 19.19071922376515, 'learning_rate': 0.11563703114697997, 'reg_lambda': 8.752617389960742, 'subsample': 0.9506154055004197, 'colsample_bytree': 0.7750335877212658}. Best is trial 1 with value: 4.945551391399786.
[I 2026-10-26 03:11:31,959] Trial 2 finished with value: 4.926209252452767 and parameters: {'max_depth': 5, 'min_child_weight': 27.418435841572773, 'learning_rate': 0.15059156715883767, 'reg_lambda': 16.5331929178097

  reject gain=0.004404626245674947 ensemble size=2

[2/2] Trying g001_c002_catboost: {'name': 'g001_c002_catboost', 'seed': 1803826929, 'family': 'catboost', 'objective': 'MAE', 'text_mode': 'selected', 'n_text': 250, 'selected_weight': 4.0, 'row_mode': 'all', 'row_fraction': 1.0, 'addons': ('geo_context',), 'comparable': 'comp_both'}


[I 2026-10-26 03:42:33,632] A new study created in RDB with name: listaware_factory_v1_g001_c002_catboost
[I 2026-10-26 03:55:23,067] Trial 0 finished with value: 4.97284088062414 and parameters: {'depth': 6, 'learning_rate': 0.04, 'l2_leaf_reg': 5}. Best is trial 0 with value: 4.97284088062414.
[I 2026-10-26 04:07:50,313] Trial 1 finished with value: 4.956974935557749 and parameters: {'depth': 10, 'learning_rate': 0.03570043900172588, 'l2_leaf_reg': 2.7405633072939373}. Best is trial 1 with value: 4.956974935557749.
[I 2026-10-26 04:12:22,719] Trial 2 finished with value: 4.904108020758023 and parameters: {'depth': 6, 'learning_rate': 0.06783905687905809, 'l2_leaf_reg': 8.838531149870558}. Best is trial 2 with value: 4.904108020758023.
[I 2026-10-26 04:24:56,207] Trial 3 finished with value: 4.885554453845851 and parameters: {'depth': 10, 'learning_rate': 0.10654557399864646, 'l2_leaf_reg': 6.721112865005687}. Best is trial 3 with value: 4.885554453845851.
[I 2026-10-26 04:29:25,732] 

  ADMIT gain=0.044055435883413985 ensemble size=3

AutoML ensemble selection: {'homes': 4351, 'mdape': 3.98791395541922, 'mape': 5.103730953258123, 'p90_ape': 10.827109247471297}
Shadow audit NOT opened; explicitly enable its separate cell.
Finished list-aware selection: {'homes': 4351, 'mdape': 3.98791395541922, 'mape': 5.103730953258123, 'p90_ape': 10.827109247471297}
Selected weights: {'reference_catboost': 0.42250000000000004, 'reference_lightgbm': 0.22749999999999998, 'g001_c002_catboost': 0.35}


In [10]:
# Inspect saved progress without opening audit or fitting anything.
factory_dir = Path(RELEASE_OUTPUT_DIR)
for name in ("progress.json", "roster.json", "summary.json", "events.json"):
    path = factory_dir / name
    if not path.is_file():
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    if name == "roster.json":
        payload = {
            "processed": len(payload.get("processed", [])),
            "admitted": [
                m["candidate"]["name"] for m in payload.get("members", [])
            ],
        }
    if name == "summary.json":
        payload = {
            "selection_metrics": payload.get("selection_metrics"),
            "weights": payload.get("effective_member_weights"),
        }
    print(name, payload)


progress.json {'stage': 'selection_complete', 'candidate': None, 'processed_count': 6, 'accepted_members': 3}
roster.json {'processed': 6, 'admitted': ['reference_catboost', 'reference_lightgbm', 'g001_c002_catboost']}
summary.json {'selection_metrics': {'homes': 4351, 'mdape': 3.98791395541922, 'mape': 5.103730953258123, 'p90_ape': 10.827109247471297}, 'weights': {'reference_catboost': 0.42250000000000004, 'reference_lightgbm': 0.22749999999999998, 'g001_c002_catboost': 0.35}}
events.json [{'generation': 0, 'candidate': {'name': 'reference_catboost', 'seed': 42, 'family': 'catboost', 'objective': 'MAE', 'text_mode': 'selected', 'n_text': 150, 'selected_weight': 4.0, 'row_mode': 'all', 'row_fraction': 1.0, 'addons': ['spacetime_splines', 'geo_context'], 'comparable': 'base'}, 'params': {'iterations': 700, 'depth': 8, 'learning_rate': 0.1570894109175486, 'l2_leaf_reg': 1.6188622243748512, 'loss_function': 'MAE', 'random_seed': 42, 'verbose': False, 'allow_writing_files': False, 'thread_

In [13]:
from pathlib import Path

RELEASE_CHECKPOINT_ARCHIVE = (
    "/content/drive/MyDrive/automl/listaware_checkpoint.zip"
)

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Mount Google Drive first.")

Path(RELEASE_CHECKPOINT_ARCHIVE).parent.mkdir(parents=True, exist_ok=True)
print(RELEASE_CHECKPOINT_ARCHIVE)

/content/drive/MyDrive/automl/listaware_checkpoint.zip


## 3 · Portable checkpoint handoff

Exactly like off-market: stop the running worker before transfer, export the output directory as a ZIP and checksum, and restore it into a new empty output folder on the other machine. Never run two processes against the same SQLite/roster state, and never keep the live SQLite DB on a mounted Drive.


In [14]:
# Explicit checkpoint transfer; both flags default to False.
EXPORT_CHECKPOINT = True
RESTORE_CHECKPOINT = False
if EXPORT_CHECKPOINT or RESTORE_CHECKPOINT:
    if not RELEASE_CHECKPOINT_ARCHIVE:
        raise ValueError("Set RELEASE_CHECKPOINT_ARCHIVE to your ZIP path")
    if EXPORT_CHECKPOINT and RESTORE_CHECKPOINT:
        raise ValueError("Export and restore cannot happen in the same run")
    if EXPORT_CHECKPOINT:
        print("Saved checkpoint:", export_checkpoint(
            RELEASE_OUTPUT_DIR, RELEASE_CHECKPOINT_ARCHIVE,
        ))
    else:
        print("Restored checkpoint:", import_checkpoint(
            RELEASE_CHECKPOINT_ARCHIVE, RELEASE_OUTPUT_DIR,
        ))


Saved checkpoint: /content/drive/MyDrive/automl/listaware_checkpoint.zip


## 4 · Freeze and evaluate (one-time)

Complete the selected number of generations first. **Opening the audit locks model membership and weights** for this output folder.

The full-history list-aware refit is a separate step after audit. It saves a model bundle under the list-aware track; **`run.py` is not modified** and does not automatically load this bundle.


In [ ]:
RUN_FULL_RELEASE_AUDIT = True
# One-way model-roster lock.
if RUN_FULL_RELEASE_AUDIT:
    frozen_factory = FactoryRun(
        settings,
        runtime_threads=profile.xgb_threads,
        checkpoint_archive=RELEASE_CHECKPOINT_ARCHIVE,
    )
    frozen_summary, audit_predictions = frozen_factory.audit()
    display(pd.DataFrame([
        {"cohort": "selection", **frozen_summary.get("selection_metrics", {})},
        {"cohort": "shadow audit", **frozen_summary.get("shadow_audit_metrics", {})},
    ]))


Frozen shadow audit: {'homes': 5951, 'mdape': 3.8917192149467033, 'mape': 4.85096578731203, 'p90_ape': 10.03586014145497}


,cohort,homes,mdape,mape,p90_ape
0,selection,4351,3.987914,5.103731,10.827109
1,shadow audit,5951,3.891719,4.850966,10.035860


In [1]:
from models.listaware_ensemble_automl_factory import (
    FactoryRun,
    FactorySettings,
)

settings = FactorySettings(
    output_dir="outputs/listaware_ensemble_release_full",
    generations=2,
    candidates_per_generation=3,
    optuna_trials=5,
    trees=700,
)

factory = FactoryRun(
    settings,
    runtime_threads=8,
)

saved_path = factory.finalize(
    model_name="list_aware_ensemble_v1"
)

print("Saved:", saved_path)

RuntimeError: Complete selection and audit before finalizing

In [ ]:
# Full-history refit of the frozen ensemble (after locked audit).
FINALIZE_RELEASE_MODEL = True
RELEASE_MODEL_NAME = "list_aware_ensemble_v1"

if FINALIZE_RELEASE_MODEL:
    finalized = FactoryRun(
        settings,
        runtime_threads=profile.xgb_threads,
        checkpoint_archive=RELEASE_CHECKPOINT_ARCHIVE,
    )
    saved_location = finalized.finalize(model_name=RELEASE_MODEL_NAME)
    print("Saved list-aware model:", saved_location)
print('test')